# 04 – E2: fine-tuning de MobileNetV2
**Diseño:** se parte de los pesos de E1 (cabeza ya entrenada), se descongela solo el último bloque de MobileNetV2 (`block_16` y la convolución final `Conv_1`), las capas BatchNormalization siguen congeladas y se entrena con learning rate 1e-5. Mismo protocolo que E0 y E1.

**Pregunta:** ¿mejora frente a E1 y a qué costo?

**Hipótesis (protocolo.md, escrita antes de ejecutar):** creemos que mejorará algunos puntos de macro-F1 frente a E1, porque las últimas capas se adaptan a patrones propios de la yuca, pero con mayor tiempo por época y riesgo de sobreajuste por el tamaño reducido del dataset.

> En Colab: Entorno de ejecución → Cambiar tipo de entorno → GPU T4. Luego "Ejecutar todas". El modelo de E1 se descarga solo desde el repositorio.

In [ ]:
# 1) Librerías y reglas del protocolo (iguales en todos los notebooks)
!pip install -q importlib_resources   # sin esto tfds.load falla en Colab

import os, time, json, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42      # semilla fija: así los resultados se pueden repetir
IMG = 224      # tamaño de entrada que esperan MobileNetV2 y ResNet50V2
BATCH = 32     # fotos por lote
EPOCHS = 10    # presupuesto máximo de épocas, igual para todos los experimentos
AUTOTUNE = tf.data.AUTOTUNE
tf.keras.utils.set_random_seed(SEED)

RESULTS = "results"
os.makedirs(RESULTS, exist_ok=True)

# Lo que ya está subido al repositorio (pesos de E1, índices excluidos, etc.) se descarga de GitHub
REPO_RAW = "https://raw.githubusercontent.com/cristianh1818/Proyecto-Yuca-Machine-learning/main/results/"

def traer(archivo):
    """Devuelve la ruta de results/<archivo>. Si no está en Colab, lo baja del repositorio."""
    ruta = f"{RESULTS}/{archivo}"
    if not os.path.exists(ruta):
        r = requests.get(REPO_RAW + archivo, timeout=600)
        r.raise_for_status()
        with open(ruta, "wb") as f:
            f.write(r.content)
    return ruta

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2) Cargar cassava con sus particiones oficiales
(ds_train, ds_val, ds_test), info = tfds.load(
    "cassava", split=["train", "validation", "test"], as_supervised=True, with_info=True)
CLASES = info.features["label"].names        # ['cbb', 'cbsd', 'cgm', 'cmd', 'healthy']
N_CLASES = len(CLASES)

# 3) Quitar exactamente las mismas fotos que en E0 y E1 (duplicados y etiquetas contradictorias).
#    Se usan los índices guardados en el EDA, así todos los experimentos ven las mismas fotos.
excluidos = json.load(open(traer("indices_excluidos_final.json")))

def filtrar(ds, indices):
    quitar = tf.constant(sorted(indices) or [-1], dtype=tf.int64)
    return (ds.enumerate()
              .filter(lambda i, xy: tf.logical_not(tf.reduce_any(tf.equal(i, quitar))))
              .map(lambda i, xy: xy))

ds_train = filtrar(ds_train, excluidos["train"])
ds_val = filtrar(ds_val, excluidos["validation"])
ds_test = filtrar(ds_test, excluidos["test"])

# 4) Pipeline: redimensionar a 224x224, guardar en memoria, barajar SOLO el entrenamiento.
#    No hay augmentation aquí: validación y prueba quedan siempre iguales.
def preparar(ds, entrenamiento=False):
    ds = ds.map(lambda x, y: (tf.cast(tf.image.resize(x, (IMG, IMG)), tf.uint8), y),
                num_parallel_calls=AUTOTUNE).cache()
    if entrenamiento:
        ds = ds.shuffle(2000, seed=SEED)
    ds = ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train = preparar(ds_train, entrenamiento=True)
val = preparar(ds_val)
test = preparar(ds_test)

# Etiquetas reales en el mismo orden que las predicciones
y_val = np.concatenate([y.numpy() for _, y in val])
y_test = np.concatenate([y.numpy() for _, y in test])
np.save(f"{RESULTS}/y_val.npy", y_val)
np.save(f"{RESULTS}/y_test.npy", y_test)
assert len(y_val) == 1876 and len(y_test) == 1874, "No coinciden las particiones con E0/E1"
print("Validación:", len(y_val), "| Prueba:", len(y_test))

In [ ]:
# 5) Funciones de evaluación (las mismas que en E0 y E1)
def evaluar(model, ds, y_true, nombre, particion):
    """Predice, calcula métricas, guarda matriz y predicciones y devuelve un diccionario."""
    t0 = time.time()
    probs = model.predict(ds, verbose=0)
    t_inf = time.time() - t0
    y_pred = probs.argmax(axis=1)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    reporte = classification_report(y_true, y_pred, target_names=CLASES, digits=3,
                                    output_dict=True, zero_division=0)
    print(f"\n===== {nombre} – {particion} =====")
    print(f"Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=CLASES, digits=3, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax, cmap="Greens", colorbar=False)
    ax.set_title(f"{nombre} – matriz de confusión ({particion})")
    plt.tight_layout(); plt.savefig(f"{RESULTS}/{nombre}_matriz_{particion}.png", dpi=120); plt.show()

    np.save(f"{RESULTS}/{nombre}_ypred_{particion}.npy", y_pred)
    np.save(f"{RESULTS}/{nombre}_probs_{particion}.npy", probs)
    return {"accuracy": round(float(acc), 4),
            "macro_f1": round(float(macro_f1), 4),
            "f1_por_clase": {c: round(reporte[c]["f1-score"], 4) for c in CLASES},
            "recall_por_clase": {c: round(reporte[c]["recall"], 4) for c in CLASES},
            "inferencia_ms_por_imagen": round(1000 * t_inf / len(y_true), 3)}

def graficar_curvas(hist, nombre):
    """Curvas de entrenamiento y validación: sirven para ver sobreajuste o subajuste."""
    h = hist.history
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    axs[0].plot(h["loss"], label="entrenamiento"); axs[0].plot(h["val_loss"], label="validación")
    axs[0].set_title("Pérdida (loss)"); axs[0].set_xlabel("época"); axs[0].legend()
    axs[1].plot(h["accuracy"], label="entrenamiento"); axs[1].plot(h["val_accuracy"], label="validación")
    axs[1].set_title("Accuracy"); axs[1].set_xlabel("época"); axs[1].legend()
    fig.suptitle(nombre); plt.tight_layout()
    plt.savefig(f"{RESULTS}/{nombre}_curvas.png", dpi=120); plt.show()

def tamano_modelo_mb(model, nombre):
    ruta = f"{RESULTS}/{nombre}.keras"
    model.save(ruta)
    return round(os.path.getsize(ruta) / 1e6, 2)

def entrenar_y_evaluar(model, nombre, lr, backbone):
    """Compila, entrena con EarlyStopping, evalúa en validación y prueba y guarda el JSON."""
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    entrenables = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    print(f"Parámetros: {model.count_params():,} | entrenables: {entrenables:,}")

    # Se detiene si la pérdida de VALIDACIÓN no mejora en 3 épocas y recupera los mejores pesos
    parar = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    t0 = time.time()
    hist = model.fit(train, validation_data=val, epochs=EPOCHS, callbacks=[parar])
    t_ent = time.time() - t0
    print(f"Entrenamiento: {t_ent / 60:.2f} min | épocas ejecutadas: {len(hist.history['loss'])}")

    graficar_curvas(hist, nombre)
    res_val = evaluar(model, val, y_val, nombre, "validacion")
    res_test = evaluar(model, test, y_test, nombre, "prueba")   # prueba: solo para reportar

    resultado = {
        "experimento": nombre,
        "config": {"seed": SEED, "img": IMG, "batch": BATCH, "epochs_max": EPOCHS,
                   "epochs_ejecutadas": len(hist.history["loss"]), "optimizador": "Adam",
                   "lr": lr, "backbone": backbone},
        "parametros": int(model.count_params()),
        "parametros_entrenables": entrenables,
        "tamano_mb": tamano_modelo_mb(model, nombre),
        "tiempo_entrenamiento_min": round(t_ent / 60, 2),
        "validacion": res_val,
        "prueba": res_test,
    }
    with open(f"{RESULTS}/{nombre}.json", "w") as f:
        json.dump(resultado, f, indent=2)
    with open(f"{RESULTS}/{nombre}_historial.json", "w") as f:
        json.dump({k: [round(float(v), 4) for v in vs] for k, vs in hist.history.items()}, f)
    print(json.dumps(resultado, indent=2))
    return resultado

# Imágenes de validación (para mostrar errores)
x_val_img = np.concatenate([x.numpy().astype("uint8") for x, _ in val])

def mostrar_errores(nombre, n=8):
    """Muestra n errores reales de VALIDACIÓN con clase real, predicha y confianza."""
    probs = np.load(f"{RESULTS}/{nombre}_probs_validacion.npy")
    pred = probs.argmax(axis=1)
    errores = np.where(pred != y_val)[0]
    print(f"Errores en validación: {len(errores)} de {len(y_val)}")
    muestra = np.random.default_rng(SEED).choice(errores, size=min(n, len(errores)), replace=False)
    fig, axs = plt.subplots(2, 4, figsize=(14, 7))
    for ax, i in zip(axs.ravel(), muestra):
        ax.imshow(x_val_img[i]); ax.axis("off")
        ax.set_title(f"#{i}\nreal: {CLASES[y_val[i]]} | pred: {CLASES[pred[i]]}\n"
                     f"confianza: {probs[i].max():.2f}", fontsize=9)
    plt.suptitle(f"{nombre} – errores (validación)"); plt.tight_layout()
    plt.savefig(f"{RESULTS}/{nombre}_errores.png", dpi=120); plt.show()
    print("Índices mostrados:", muestra.tolist())

## Punto de partida: E1
Se carga E1 y se comprueba que da el mismo macro-F1 que en el notebook 03 (0,6722).

In [ ]:
NOMBRE_E1 = "E1_mobilenetv2_congelado"
e1 = tf.keras.models.load_model(traer(f"{NOMBRE_E1}.keras"))
p = e1.predict(val, verbose=0).argmax(axis=1)
print("Macro-F1 de E1 en validación:", round(f1_score(y_val, p, average="macro"), 4))

## Descongelar el último bloque
Cambiar `CAPAS_A_DESCONGELAR` si se quiere probar con más o menos capas.

In [ ]:
NOMBRE = "E2_mobilenetv2_finetuning"
LR = 1e-5                                   # tasa menor para no borrar lo aprendido en ImageNet
CAPAS_A_DESCONGELAR = ("block_16", "Conv_1")

e2 = tf.keras.models.load_model(traer(f"{NOMBRE_E1}.keras"))     # copia nueva de E1
base = [l for l in e2.layers if isinstance(l, tf.keras.Model)][0]   # el MobileNetV2 interno
base.trainable = True
for capa in base.layers:
    es_bloque_final = capa.name.startswith(CAPAS_A_DESCONGELAR)
    es_batchnorm = isinstance(capa, tf.keras.layers.BatchNormalization)
    capa.trainable = es_bloque_final and not es_batchnorm           # BN siempre congelada

print("Capas descongeladas:", [c.name for c in base.layers if c.trainable])

## Entrenar y evaluar

In [ ]:
r_e2 = entrenar_y_evaluar(e2, NOMBRE, LR,
                          "MobileNetV2 (ImageNet), fine-tuning de block_16 y Conv_1 desde E1; BN congelada")

In [ ]:
mostrar_errores(NOMBRE)

## Comparación E1 vs E2 (validación)

In [ ]:
r_e1 = json.load(open(traer(f"{NOMBRE_E1}.json")))
filas = [("Accuracy", "accuracy"), ("Macro-F1", "macro_f1")]
print(f"{'métrica':28s}{'E1':>10s}{'E2':>10s}")
for txt, k in filas:
    print(f"{txt:28s}{r_e1['validacion'][k]:10.4f}{r_e2['validacion'][k]:10.4f}")
for c in CLASES:
    print(f"{'F1 ' + c:28s}{r_e1['validacion']['f1_por_clase'][c]:10.4f}{r_e2['validacion']['f1_por_clase'][c]:10.4f}")
print(f"{'Parámetros entrenables':28s}{r_e1['parametros_entrenables']:10d}{r_e2['parametros_entrenables']:10d}")
print(f"{'Entrenamiento (min)':28s}{r_e1['tiempo_entrenamiento_min']:10.2f}{r_e2['tiempo_entrenamiento_min']:10.2f}")
print(f"{'Inferencia (ms/img, val)':28s}{r_e1['validacion']['inferencia_ms_por_imagen']:10.2f}{r_e2['validacion']['inferencia_ms_por_imagen']:10.2f}")

## Análisis
[COMPLETAR TRAS EJECUTAR] — ver `analisis_E2.md`.

In [ ]:
# Descargar los resultados para subirlos a la carpeta results/ del repositorio.
# Los .keras grandes (más de 25 MB) no se pueden subir por la web de GitHub: se excluyen.
import glob, zipfile
with zipfile.ZipFile("results_E2.zip", "w") as z:
    for f in glob.glob(f"{RESULTS}/*"):
        if f.endswith(".keras") and os.path.getsize(f) > 25e6:
            print("Se omite por tamaño:", f)
            continue
        z.write(f)
from google.colab import files
files.download("results_E2.zip")